In [17]:
import pandas as pd
import numpy as np
from scipy.interpolate import UnivariateSpline

df = pd.read_csv('../../Data/Final/final_CHANGI_long.csv')
print(df.head())

            x         y        period    avg_LST
0  103.964566  1.350459  Mar-Apr 2000  23.775064
1  103.964566  1.351269  Mar-Apr 2000  23.403744
2  103.964566  1.352079  Mar-Apr 2000  22.966872
3  103.964566  1.352889  Mar-Apr 2000  22.417757
4  103.965377  1.348838  Mar-Apr 2000  23.699749


In [10]:
# Extract unique (x, y) pairs
unique_xy = df[['x', 'y']].drop_duplicates().reset_index(drop=True)
print(f"Total unique (x, y) pairs: {len(unique_xy)}")
print(unique_xy.head())  # Display first few rows

# Extract unique periods
unique_periods = df['period'].unique()
total_periods = len(unique_periods)

# Create expected full grid of (x, y, period)
expected_grid_size = len(unique_xy) * total_periods

# Count actual rows
actual_rows = len(df)

# Check for missing (x, y, period) combinations
if actual_rows < expected_grid_size:
    print("There are missing (x, y, period) combinations.")
else:
    print("No missing (x, y, period) combinations.")

Total unique (x, y) pairs: 5290
            x         y
0  103.964566  1.350459
1  103.964566  1.351269
2  103.964566  1.352079
3  103.964566  1.352889
4  103.965377  1.348838
No missing (x, y, period) combinations.


In [16]:
# Extract first and last years from dataset
start_year = int(unique_periods[0][-4:])
end_year = int(unique_periods[-1][-4:])

# Generate expected periods dynamically
months = ['Jan-Feb', 'Mar-Apr', 'May-Jun', 'Jul-Aug', 'Sep-Oct', 'Nov-Dec']
expected_periods = [f"{m} {y}" for y in range(start_year, end_year + 1) for m in months]

# Find missing periods
missing_periods = sorted(set(expected_periods) - set(unique_periods))

print(f"Missing periods: {missing_periods}")
print("Number of missing periods:", len(missing_periods))

Missing periods: ['Jan-Feb 2000', 'Jul-Aug 2008', 'Jul-Aug 2010', 'Mar-Apr 2008', 'May-Jun 2005', 'May-Jun 2010', 'May-Jun 2011', 'Nov-Dec 2001', 'Nov-Dec 2008', 'Nov-Dec 2009', 'Nov-Dec 2022', 'Sep-Oct 2003', 'Sep-Oct 2009', 'Sep-Oct 2010']
Number of missing periods: 14


In [ ]:
# Load dataset
existing_df = pd.read_csv('../../Data/Final/final_CHANGI_long.csv')
unique_xy = pd.read_csv('../../Data/Final/final_CHANGI_wide.csv', usecols=['x', 'y']).drop_duplicates()

# Define missing periods
missing_periods = ['Jul-Aug 2008', 'Jul-Aug 2010', 'Mar-Apr 2008', 'May-Jun 2005', 'May-Jun 2010', 'May-Jun 2011', 
                   'Nov-Dec 2001', 'Nov-Dec 2008', 'Nov-Dec 2009', 'Nov-Dec 2022', 'Sep-Oct 2003', 'Sep-Oct 2009', 'Sep-Oct 2010']

# Create DataFrame for missing periods
missing_entries = pd.DataFrame([(x, y, period) for x, y in zip(unique_xy['x'], unique_xy['y']) for period in missing_periods],
                               columns=['x', 'y', 'period'])

# Append missing entries to existing data
filled_df = pd.concat([existing_df, missing_entries], ignore_index=True).sort_values(by=['period', 'x', 'y'])

# Save to CSV
filled_df.to_csv('../../Data/Final/Imputation/filled_CHANGI_long.csv', index=False)

print("Missing periods successfully filled and saved!")


Missing periods successfully filled and saved!


In [ ]:
# Load the filled dataset
filled_df = pd.read_csv('filled_CHANGI_long.csv')

# Rename columns
filled_df.rename(columns={'period': 'Date', 'avg_LST': 'Value'}, inplace=True)

# Create a mapping of Date to a sequential index
unique_dates = sorted(filled_df['Date'].unique())  # Ensure chronological order
date_mapping = {date: i for i, date in enumerate(unique_dates)}
filled_df['date_index'] = filled_df['Date'].map(date_mapping)

# Function to apply spline interpolation
def spline_impute(group):
    if group['Value'].isnull().all():  # If all values are missing, return original
        return group
    
    valid_data = group.dropna(subset=['Value'])  # Keep only non-null values
    if len(valid_data) < 4:  # Spline needs at least 4 points (degree + 1)
        return group
    
    # Fit cubic spline
    spline = UnivariateSpline(valid_data['date_index'], valid_data['Value'], k=3, s=0)
    
    # Predict missing values
    missing_mask = group['Value'].isna()
    group.loc[missing_mask, 'Value'] = spline(group.loc[missing_mask, 'date_index'])
    
    return group

# Apply spline interpolation group-wise
filled_df = filled_df.groupby(['x', 'y'], group_keys=False).apply(spline_impute)

# Drop helper column
filled_df.drop(columns=['date_index'], inplace=True)

# Save the imputed dataset
filled_df.to_csv('imputed_CHANGI_long.csv', index=False)

print("Spline interpolation completed and saved as 'imputed_CHANGI_long.csv'!")

Spline interpolation completed and saved as 'imputed_CHANGI_long.csv'!
